# Persistent mixtures and saliency

Warm-start Gaussian mixtures across frames and maintain an exponential moving average of component weights. Compare selection rules, decay constants and frame windows.


## 1. Setup

Load shared helpers and PersistentGMM from foveanet.py.


In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import foveanet as fn

dataset = fn.load_gesture()
events, label = dataset[0]
x, y, t, p = fn.unpack(events)
frames = list(fn.iter_frames(x, y, t))

print("recordings:", len(dataset))
print("frames in this recording:", len(frames))
print("window:", fn.WINDOW_US / 1000, "ms | k =", fn.K)
print("decay:", fn.DECAY, "| hysteresis:", fn.HYSTERESIS, "| reg_covar:", fn.REG_COVAR)

## 2. Independent fits

Measure component-index switches when each frame is fitted independently. Component labels from separate fits need not correspond to the same region.


In [ ]:
def index_run(recording, warm):
    xr, yr, tr = recording
    pg = fn.PersistentGMM(warm=warm)
    ids = []
    for s, xs, ys, ts in fn.iter_frames(xr, yr, tr):
        coords = np.column_stack([xs, ys]).astype(np.float32)
        pg.update(coords)
        ids.append(int(pg.model.weights_.argmax()))
    return ids


rec = (x, y, t)
for warm in [False, True]:
    ids = index_run(rec, warm)
    mean_run, longest, switch = fn.dwell_stats(ids)
    name = "warm-started" if warm else "fresh fit each frame"
    print(
        f"{name:>22}: mean run {mean_run:5.1f} frames, longest {longest:3d}, "
        f"index changes on {100 * switch:.0f}% of frames"
    )

## 3. Warm-started fits

Initialise each fit from the preceding means, weights and precisions. This encourages component continuity but does not guarantee identity. reg_covar prevents degenerate covariance matrices. Measure runtime rather than assuming fewer iterations.


## 4. Persistence

Update each component score as decay * previous_score + (1 - decay) * current_weight. Plot the scores across the recording.


In [ ]:
pg = fn.PersistentGMM()
for s, xs, ys, ts in frames:
    pg.update(np.column_stack([xs, ys]).astype(np.float32))

hist = np.array(pg.history)
ms = (np.array([f[0] for f in frames]) - t.min()) / 1000

fig, ax = plt.subplots(figsize=(11, 3.6))
for cid in range(pg.k):
    ax.plot(ms, hist[:, cid], lw=1.8, label=f"component {cid}")
ax.set_xlabel("time (ms)")
ax.set_ylabel("persistence")
ax.set_title("Persistence of each mixture component over the recording")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

print("final persistence:", pg.persistence.round(3))
print("final mixture weights:", pg.model.weights_.round(3))

## 5. Saliency map

Render mixture density on the sensor grid, weighted by accumulated persistence. Compare this map with the selected percentile box.


In [ ]:
SHOW = len(frames) // 2

pg = fn.PersistentGMM()
accum = np.zeros(fn.SENSOR)
snapshot = None
for i, (s, xs, ys, ts) in enumerate(frames):
    coords = np.column_stack([xs, ys]).astype(np.float32)
    pg.update(coords)
    smap = pg.saliency_map(rule="persistence")
    accum = fn.DECAY * accum + (1 - fn.DECAY) * fn.normalise(smap)
    if i == SHOW:
        cid = pg.select(rule="persdens+hyst")
        snapshot = (
            fn.frame_image(xs, ys),
            pg.saliency_map(rule="size"),
            smap,
            accum.copy(),
            pg.component_box(cid, coords),
        )

img, map_size, map_pers, map_acc, box = snapshot
titles = [
    "events in frame",
    "saliency from mixture weight",
    "saliency from persistence",
    "persistence accumulated to here",
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4.3))
for ax, data, title in zip(axes, [img, map_size, map_pers, map_acc], titles):
    ax.imshow(data, cmap="hot" if title.startswith("events") else "magma")
    ax.set_title(title, fontsize=10)
    ax.axis("off")
bx0, by0, bx1, by1 = box
axes[2].add_patch(
    patches.Rectangle((bx0, by0), bx1 - bx0, by1 - by0, lw=2, edgecolor="cyan", facecolor="none")
)
plt.tight_layout()
plt.show()

## 6. Selection rules

Compare instantaneous size/density with accumulated persistence/persistence-density. Measure dwell time, switching, motion, event capture and crop area.


In [ ]:
def evaluate(recs, rule, warm=True, decay=fn.DECAY):
    dwell, longest, switch, jump, capture, boxfrac = [], [], [], [], [], []
    for xr, yr, tr in recs:
        pg = fn.PersistentGMM(warm=warm, decay=decay)
        ids, centres, caps, bfs = [], [], [], []
        for s, xs, ys, ts in fn.iter_frames(xr, yr, tr):
            coords = np.column_stack([xs, ys]).astype(np.float32)
            pg.update(coords)
            cid = pg.select(rule=rule)
            box = pg.component_box(cid, coords)
            if box is None:
                continue
            ids.append(cid)
            centres.append(fn.box_centre(box))
            caps.append(((xs >= box[0]) & (xs <= box[2]) & (ys >= box[1]) & (ys <= box[3])).mean())
            bfs.append(
                max((box[2] - box[0]) * (box[3] - box[1]), 1) / (fn.SENSOR[0] * fn.SENSOR[1])
            )
        if len(ids) < 3:
            continue
        d, l, s = fn.dwell_stats(ids)
        c = np.array(centres)
        dwell.append(d)
        longest.append(l)
        switch.append(s)
        jump.append(np.hypot(*(c[1:] - c[:-1]).T).mean())
        capture.append(np.mean(caps))
        boxfrac.append(np.mean(bfs))
    cap, bf = np.mean(capture), np.mean(boxfrac)
    return (np.mean(dwell), np.mean(longest), np.mean(switch), np.mean(jump), cap, bf, cap / bf)


rng = np.random.default_rng(0)
targets = np.array(dataset.targets)
all_idx = np.arange(len(dataset))
sample = []
for cls in sorted(set(targets)):
    sample += rng.choice(all_idx[targets == cls], 2, replace=False).tolist()

recs = []
for i in sample:
    ev, _ = dataset[i]
    xi, yi, ti, _ = fn.unpack(ev)
    recs.append((xi, yi, ti))
print(f"recordings: {len(recs)}\n")

rules = [
    ("cold mixture, size", dict(rule="size", warm=False)),
    ("warm mixture, size", dict(rule="size")),
    ("warm mixture, density", dict(rule="density")),
    ("warm mixture, persistence", dict(rule="persistence")),
    ("warm, persistence + hysteresis", dict(rule="persistence+hyst")),
    ("warm, persistence x density", dict(rule="persdens")),
    ("warm, persistence x density + hyst", dict(rule="persdens+hyst")),
]
print(
    f"{'fovea rule':>36}{'dwell':>7}{'longest':>9}{'switch':>9}"
    f"{'jump':>7}{'capture':>9}{'boxfrac':>9}{'concentr':>10}"
)
for name, kw in rules:
    d, l, s, j, c, b, k = evaluate(recs, **kw)
    print(f"{name:>36}{d:>7.1f}{l:>9.1f}{100 * s:>8.1f}%{j:>7.1f}{c:>9.2f}{b:>9.3f}{k:>10.1f}")

## Interpreting selection metrics

Read capture and area together. Longer dwell does not establish better task performance; classification and annotated tracking are evaluated separately.


## 7. Decay sweep

Vary the persistence decay and compare stability with event capture. Larger decay retains a longer history and can increase lag.


In [ ]:
print(f"{'decay':>8}{'dwell':>8}{'longest':>9}{'switch':>9}{'jump':>7}{'capture':>9}{'boxfrac':>9}")
for decay in [0.0, 0.5, 0.7, 0.85, 0.95]:
    d, l, s, j, c, b, _ = evaluate(recs, rule="persistence+hyst", decay=decay)
    print(f"{decay:>8.2f}{d:>8.1f}{l:>9.1f}{100 * s:>8.1f}%{j:>7.1f}{c:>9.2f}{b:>9.3f}")

## 8. Frame-window sweep

Measure fit runtime and component stability over a range of microsecond windows. Sparse windows may not contain enough events to fit the requested components.


In [ ]:
import warnings

SPAN_US = 1_500_000
WINDOWS = [1_000, 2_000, 5_000, 10_000, 20_000, 33_000, 50_000, 75_000, 100_000]


def frame_length_run(x, y, t, window, warm, k=fn.K, rule="persdens+hyst"):
    m = (t >= t.min()) & (t < t.min() + SPAN_US)
    x, y, t = x[m], y[m], t[m]
    pg = fn.PersistentGMM(k=k, warm=warm)
    ids, caps, bfs, cents, nev = [], [], [], [], []
    starts = np.arange(t.min(), t.max(), window)
    dropped, elapsed = 0, 0.0
    for s in starts:
        sel = (t >= s) & (t < s + window)
        n = int(sel.sum())
        if n < k:
            dropped += 1
            continue
        xs, ys = x[sel], y[sel]
        coords = np.column_stack([xs, ys]).astype(np.float32)
        c0 = time.perf_counter()
        pg.update(coords)
        cid = pg.select(rule=rule)
        elapsed += time.perf_counter() - c0
        box = pg.component_box(cid, coords)
        if box is None:
            dropped += 1
            continue
        ids.append(cid)
        nev.append(n)
        caps.append(((xs >= box[0]) & (xs <= box[2]) & (ys >= box[1]) & (ys <= box[3])).mean())
        bfs.append(max((box[2] - box[0]) * (box[3] - box[1]), 1) / (fn.SENSOR[0] * fn.SENSOR[1]))
        cents.append(fn.box_centre(box))
    if len(ids) < 5:
        return None
    d, l, sw = fn.dwell_stats(ids)
    c = np.array(cents)
    sec = window / 1e6
    ms_per_frame = 1000 * elapsed / len(ids)
    cap, bf = np.mean(caps), np.mean(bfs)
    return dict(
        ev=np.mean(nev),
        drop=dropped / max(len(starts), 1),
        dwell_s=d * sec,
        longest_s=l * sec,
        switch_hz=sw / sec,
        px_per_ms=np.hypot(*(c[1:] - c[:-1]).T).mean() / (window / 1000),
        cap=cap,
        bf=bf,
        conc=cap / bf,
        ms=ms_per_frame,
        realtime=(window / 1e3) / ms_per_frame,
    )


frame_rng = np.random.default_rng(1)
frame_sample = [int(frame_rng.choice(all_idx[targets == cls])) for cls in sorted(set(targets))][:8]
frame_recs = []
for i in frame_sample:
    ev, _ = dataset[i]
    xi, yi, ti, _ = fn.unpack(ev)
    frame_recs.append((xi, yi, ti))
print(f"{len(frame_recs)} recordings, one per class, first {SPAN_US / 1e6:.1f} s of each\n")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for warm in [True, False]:
        print("warm-started mixture" if warm else "\nfresh mixture every frame")
        print(
            f"{'window':>8}{'ev/frame':>10}{'dropped':>9}{'dwell s':>9}{'longest s':>11}"
            f"{'switch/s':>10}{'px/ms':>8}{'capture':>9}{'boxfrac':>9}{'concentr':>10}"
            f"{'ms/frame':>10}{'realtime':>10}"
        )
        for w in WINDOWS:
            got = [frame_length_run(*r, w, warm) for r in frame_recs]
            got = [g for g in got if g]
            if not got:
                print(f"{w / 1000:>7.0f}ms   no usable frames")
                continue
            avg = lambda key: np.mean([g[key] for g in got])
            rt = avg("realtime")
            note = "" if rt >= 1 else "   cannot keep up"
            print(
                f"{w / 1000:>7.0f}ms{avg('ev'):>10.0f}{100 * avg('drop'):>8.0f}%"
                f"{avg('dwell_s'):>9.2f}{avg('longest_s'):>11.2f}{avg('switch_hz'):>10.1f}"
                f"{avg('px_per_ms'):>8.2f}{avg('cap'):>9.2f}{avg('bf'):>9.3f}"
                f"{avg('conc'):>10.1f}{avg('ms'):>10.2f}{rt:>10.1f}{note}"
            )

## Choosing a frame window

Compare processing time with the frame duration and inspect event counts. The shared exploratory default is 50 ms; report crop experiments explicitly use 10 ms.


## 9. Full recording

Plot the selected region over time using the chosen rule and inspect both continuity and spatial coverage.


In [ ]:
pg = fn.PersistentGMM()
timeline = []
for s, xs, ys, ts in frames:
    coords = np.column_stack([xs, ys]).astype(np.float32)
    pg.update(coords)
    cid = pg.select(rule="persdens+hyst")
    box = pg.component_box(cid, coords)
    if box is None:
        continue
    timeline.append((s, pg.saliency_map(rule="persistence"), box, cid, float(pg.persistence[cid])))

ids = [e[3] for e in timeline]
mean_run, longest, switch = fn.dwell_stats(ids)
print(
    f"frames: {len(timeline)}, mean dwell {mean_run:.1f}, longest run {longest}, "
    f"switches on {100 * switch:.1f}% of frames"
)

pick = np.linspace(0, len(timeline) - 1, 6).astype(int)
fig, axes = plt.subplots(2, 3, figsize=(11, 7))
for ax, i in zip(axes.ravel(), pick):
    s, smap, box, cid, pers = timeline[i]
    ax.imshow(smap, cmap="magma")
    bx0, by0, bx1, by1 = box
    ax.add_patch(
        patches.Rectangle(
            (bx0, by0), bx1 - bx0, by1 - by0, lw=2, edgecolor="cyan", facecolor="none"
        )
    )
    ax.set_title(
        f"t = {(s - t.min()) / 1000:.0f} ms, component {cid}, persistence {pers:.2f}", fontsize=9
    )
    ax.axis("off")
plt.suptitle("Persistence-weighted saliency and the fovea it selects")
plt.tight_layout()
plt.show()

## 10. Related experiments

Notebook 04 tests other recordings. Notebook 05 summarises classification; notebook 06 evaluates target tracking.
